Steps:
1. Import packages needed
2. Input data (includes train test split, normalize)
3. Create model with Hyperparameter analysis
4. Acquire parameters / Evaluate model
5. Visualizations (Cluster Maps)

## 1. Import Packages

In [35]:
import matplotlib.pyplot as plt
import altair as alt
import numpy as np
import pandas as pd

In [36]:
np.set_printoptions(precision=5)
random_state = 42
import sklearn

#One hot encoding packages
from ast import literal_eval
from sklearn.preprocessing import MultiLabelBinarizer

# packages for visualizations
import umap

#packages for hypertuning
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score

#packages for clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

## 2. Input Data

In [37]:
#import data from the scryfall data set and extract creature data
df = pd.read_csv('./data/final-scryfall-unique-artwork.csv')
creature_df = df[df['type'].str.contains('Creature')]
creature_df.head()

,id,oracle_id,name,set_id,set,set_name,artist_ids,artist,released_at,type_line,...,power,toughness,edhrec_rank,type,subtype,legality_commander,legality_standard,price_usd,image_uri_normal,image_uri_art_crop
1,0000579f-7b35-4ed3-b44c-db2a538066fe,44623693-51d6-49ad-8cd7-140505caf02f,Fury Sliver,c1d109bc-ffd8-428f-8d7d-3f8d7e648046,tsp,Time Spiral,['d48dd097-720d-476a-8722-6a02854ae28b'],Paolo Parente,2006-10-06,Creature — Sliver,...,3,3,9808.0,Creature,Sliver,legal,not_legal,0.46,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
2,00006596-1166-4a79-8443-ca9f82e6db4e,8ae3562f-28b7-4462-96ed-be0cf7052ccc,Kor Outfitter,eb16a2bd-a218-4e4e-8339-4aa1afc0c8d2,zen,Zendikar,['aa7e89ed-d294-4633-9057-ce04dacfcfa4'],Kieran Yanner,2009-10-02,Creature — Kor Soldier,...,2,2,19672.0,Creature,Kor Soldier,legal,not_legal,0.11,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
3,0000cd57-91fe-411f-b798-646e965eec37,9f0d82ae-38bf-45d8-8cda-982b6ead1d72,Siren Lookout,fe0dad85-54bc-4151-9200-d68da84dd0f2,xln,Ixalan,['a8e7b854-b15a-421a-b66d-6e68187ae285'],Chris Rallis,2017-09-29,Creature — Siren Pirate,...,1,2,18843.0,Creature,Siren Pirate,legal,not_legal,0.04,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
4,0001f1ef-b957-4a55-b47f-14839cdbab6f,ef027846-be81-4959-a6b5-56bd01b1e68a,Venerable Knight,a90a7b2f-9dd8-4fc7-9f7d-8ea2797ec782,eld,Throne of Eldraine,['9c201dbe-db56-429a-87e6-189ea70c2632'],Colin Boyer,2019-10-04,Creature — Human Knight,...,2,1,18404.0,Creature,Human Knight,legal,not_legal,0.15,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
6,0002ab72-834b-4c81-82b1-0d2760ea96b0,645b5784-a6f7-4cf3-966a-e1a51420b96b,Mystic Skyfish,bc94aba1-7376-4e02-a12d-3a2efb66ab0f,m21,Core Set 2021,['bb677b1a-ce51-4888-83d6-5a94de461ff9'],Alayna Danner,2020-07-03,Creature — Fish,...,3,1,23732.0,Creature,Fish,legal,not_legal,0.10,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...


In [38]:
# calculate thresholds for top 1%, bottom 1%, and middle 1% by price (acquired from MDS artowrk file)
creature_df.loc[:,'price_usd'] = creature_df['price_usd'].round(2)

top_1 = creature_df['price_usd'].quantile(0.99)
bottom_1 = creature_df['price_usd'].quantile(0.01)
mid_lower = creature_df['price_usd'].quantile(0.495) # just below median
mid_upper = creature_df['price_usd'].quantile(0.505) # just above median

top_1_df = creature_df[creature_df['price_usd'] >= top_1]
top_1_df['percentile'] = 'Top 1%'

bottom_1_df = creature_df[creature_df['price_usd'] <= bottom_1]
bottom_1_df['percentile'] = 'Bottom 1%'

mid_1_df = creature_df[(creature_df['price_usd'] >= mid_lower) & (creature_df['price_usd'] < mid_upper)]
mid_1_df['percentile'] = 'Middle 1%'

percentile_df = pd.concat([top_1_df, bottom_1_df, mid_1_df]).reset_index()

C:\Users\jaymj\AppData\Local\Temp\ipykernel_8116\2941743629.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_1_df['percentile'] = 'Top 1%'
C:\Users\jaymj\AppData\Local\Temp\ipykernel_8116\2941743629.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bottom_1_df['percentile'] = 'Bottom 1%'
C:\Users\jaymj\AppData\Local\Temp\ipykernel_8116\2941743629.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value inste

In [39]:
#one hot encoding of labels (acquired from MDS artowrk file)
one_hot_encoding_df = percentile_df.copy()
one_hot_encoding_df['subtype'] = percentile_df['subtype'].fillna('').str.split()

mlb = MultiLabelBinarizer()
encoded_subtypes = mlb.fit_transform(one_hot_encoding_df['subtype'])
subtypes_df = pd.DataFrame(
    encoded_subtypes,
    columns='s_'+mlb.classes_,
    index=one_hot_encoding_df.index
    )

encoded_keywords = mlb.fit_transform(one_hot_encoding_df['keywords'].apply(literal_eval))
keywords_df = pd.DataFrame(
    encoded_keywords,
    columns='k_'+mlb.classes_,
    index=one_hot_encoding_df.index
    )

encoded_colors = mlb.fit_transform(one_hot_encoding_df['color_identity'].apply(literal_eval))
colors_df = pd.DataFrame(
    encoded_colors,
    columns='c_'+mlb.classes_,
    index=one_hot_encoding_df.index
    )

features = pd.concat([subtypes_df, keywords_df, colors_df], axis=1)

## 3. Create model

In [40]:
# Create function to return score for clustering
def silhouette_scorer(estimator, X):
    #input: model - model used for score
    #input: X - pandas DataFrame of X features used to train model
    #return: score - Silhouette score float

    labels = estimator.predict(X)
    return silhouette_score(X, labels)

def daviesbloudin_scorer(estimator, X):
    #input: model - model used for score
    #input: X - pandas DataFrame of X features used to train model
    #return: score - Davies-Bloudin score float

    labels = estimator.predict(X)
    return davies_bouldin_score(X, labels)

def calinskiharabasz_scorer(estimator, X):
    #input: model - model used for score
    #input: X - pandas DataFrame of X features used to train model
    #return: score - Calinski-Harabasz score float
    labels = estimator.predict(X)
    return calinski_harabasz_score(X, labels)

In [41]:
#Choose parameters to hypertune
params ={
    'umap__n_neighbors': [10, 15, 20],
    'umap__min_dist': [0.1, 0.5, 0.75],
    'umap__metric': ['jaccard','hamming'],
    'kmeans__n_clusters': [4, 6, 8]
}

#Create pipeline to acquire Umap positions than apply K-mean Clusters
pipeline= Pipeline([
    ('umap', umap.UMAP(random_state=random_state)),
    ('kmeans', KMeans(random_state=random_state))
])

#hypertune using parameters above and score based on silhoutte
grid_search = GridSearchCV(
    pipeline,
    params,
    scoring=silhouette_scorer,
    cv=5
)

grid_search.fit(features)

c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1887: UserWarning: gradient function is not yet implemented for jaccard distance metric; inverse_transform will be unavailable
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:127: UserWarning: A few of your vertices were disconnected from the manifold.  This shouldn't cause problems.
Disconnection_distance = 1 has removed 317138 edges.
It has only fully disconnected 2 vertices.
Use umap.utils.disconnected_vertices() to identify them.
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1887: UserWarning: gradient function is not yet implemented for jaccard distance metric; inverse_transform will be unavailable
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting r

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'kmeans__n_clusters': [4, 6, ...], 'umap__metric': ['jaccard', 'hamming'], 'umap__min_dist': [0.1, 0.5, ...], 'umap__n_neighbors': [10, 15, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",<function sil...002756582DF80>
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1

In [42]:
#Results of best model with parameters
best_params = grid_search.best_params_
best_estimator = grid_search.best_estimator_
best_score = grid_search.best_index_
print("Best Parameters:", grid_search.best_params_)
print("Best Score (Silhouette):", grid_search.best_score_)

Best Parameters: {'kmeans__n_clusters': 6, 'umap__metric': 'hamming', 'umap__min_dist': 0.1, 'umap__n_neighbors': 20}
Best Score (Silhouette): 0.09241826453489463


In [43]:
#Creating coordinates for U-MAP model using parameters from hypertuning
best_k = best_params['kmeans__n_clusters']
umap_clf = umap.UMAP(metric=best_params['umap__metric'], min_dist=best_params['umap__min_dist'], n_neighbors=best_params['umap__n_neighbors'], random_state=random_state)
umap_pos = umap_clf.fit_transform(features)
umap_percentile_df = percentile_df.copy()
umap_percentile_df['x'] = [x[0] for x in umap_pos]
umap_percentile_df['y'] = [y[1] for y in umap_pos]

kmeans_umap = KMeans(n_clusters = best_k, random_state=random_state)
kmeans_umap.fit(umap_pos)

#Acquire scores using cross validation split
silhoue_scores = cross_val_score(
    kmeans_umap,
    umap_pos,
    scoring=silhouette_scorer,
    cv=3
    )
daviesbl_scores = cross_val_score(
    kmeans_umap,
    umap_pos,
    scoring=daviesbloudin_scorer,
    cv=3
    )
calinski_scores = cross_val_score(
    kmeans_umap,
    umap_pos,
    scoring=calinskiharabasz_scorer,
    cv=3
    )

print(f'Silhouette score for {best_k} clusters: {silhoue_scores.mean()} ({silhoue_scores.std()})')
print(f'Davies-Bouldin score for {best_k} clusters: {daviesbl_scores.mean()} ({daviesbl_scores.std()})')
print(f'Calinski-Harabasz score for {best_k} clusters: {calinski_scores.mean()} ({calinski_scores.std()})')

c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Silhouette score for 6 clusters: 0.6662901242574056 (0.01726827610898092)
Davies-Bouldin score for 6 clusters: 0.48973587437490423 (0.018820500182611673)
Calinski-Harabasz score for 6 clusters: 1229.9159045006245 (215.47209979308795)


## 5. Visualizations

In [44]:
#Create plot to incorporate artwork in cluster maps
def genPlot(df, percentile=True, sample_n=200, art_crop=True):
    # input: df -- dataframe (augmented with the x/y columns)
    # input: sample_n -- number of cards to display from each percentile, randomly sampled
    # input: art_crop -- True to use cropped art image, False to use full card image
    # return: an altair chart (e.g., return alt.Chart(...))

    sampled_df = df.groupby('percentile').apply(lambda x: x.sample(n=sample_n, random_state=random_state)).rename(columns={'percentile':'sample_percentile'}).reset_index()

    if percentile == True:
        squares = alt.Chart(sampled_df).mark_square(size=2000).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        color=alt.Color('percentile:N', legend=alt.Legend(title="Percentiles", orient='top-left', labelFontSize=16, titleFontSize=20))
        )

    else:
        squares = alt.Chart(sampled_df).mark_square(size=2000).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        color=alt.Color('labels:N', legend=alt.Legend(title="K-Cluster Label", orient='top-left', labelFontSize=16, titleFontSize=20))
        )

    image_url = 'image_uri_art_crop' if art_crop else 'image_uri_normal'
    
    images = alt.Chart(sampled_df).mark_image(width=40, height=40).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        url=image_url,
        tooltip=[
            alt.Tooltip("name"),
            alt.Tooltip("subtype"),
            alt.Tooltip("keywords"),
            alt.Tooltip("color_identity"),
            alt.Tooltip("price_usd")
        ]
    ).properties(
        width=800,
        height=800,
        title="UMAP Color-Coded by Price Bracket"
    )

    cluster_points = alt.Chart(sampled_df).mark_point(filled=True,size=100).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        color=alt.Color('labels:N', legend=alt.Legend(title="K-Cluster Label", orient='top-left', labelFontSize=16, titleFontSize=20)),
        tooltip=[
            alt.Tooltip("name"),
            alt.Tooltip("subtype"),
            alt.Tooltip("keywords"),
            alt.Tooltip("color_identity"),
            alt.Tooltip("price_usd")
        ]
    ).properties(
        width=800,
        height=800,
        title="UMAP with K-Means Clustering (w/o images)",
    )

    price_points = alt.Chart(sampled_df).mark_point(filled=True,size=100).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        color=alt.Color('percentile:N', legend=alt.Legend(title="Percentiles", orient='top-left', labelFontSize=16, titleFontSize=20)),
        tooltip=[
            alt.Tooltip("name"),
            alt.Tooltip("subtype"),
            alt.Tooltip("keywords"),
            alt.Tooltip("color_identity"),
            alt.Tooltip("price_usd")
        ]
    ).properties(
        width=800,
        height=800,
        title="UMAP Color-Coded by Price Bracket (w/o images)",
    )
    
    return sampled_df, (squares + images).configure_title(fontSize=30), cluster_points.configure_title(fontSize=30), price_points.configure_title(fontSize=30)

In [45]:
#UMAP visualization
umap_percentile_df['labels'] = kmeans_umap.labels_
umap_price_df, umap_price_image, umap_clusters, umap_price = genPlot(umap_percentile_df, False, 100)
umap_price_image

C:\Users\jaymj\AppData\Local\Temp\ipykernel_8116\4182680683.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.groupby('percentile').apply(lambda x: x.sample(n=sample_n, random_state=random_state)).rename(columns={'percentile':'sample_percentile'}).reset_index()


alt.LayerChart(...)

In [46]:
#UMAP visualization without pictures and color coded by price and cluster label
umap_clusters.show()
umap_price.show()

alt.Chart(...)

alt.Chart(...)

In [47]:
#Hypertuning Sensitivity Analysis
hypertuning_best_k = [
    (best_params['kmeans__n_clusters']-2),
    (best_params['kmeans__n_clusters']-1),
    best_params['kmeans__n_clusters'],
    (best_params['kmeans__n_clusters']+1),
    (best_params['kmeans__n_clusters']+2)]
sil_scores = []
sil_std = []
dav_scores = []
dav_std = []
cal_scores = []
cal_std = []

for k in hypertuning_best_k:
    hypertuning_umap_clf = umap.UMAP(
        metric=best_params['umap__metric'],
        min_dist=best_params['umap__min_dist'],
        n_neighbors=best_params['umap__n_neighbors'],
        random_state=random_state
        )
    hypertuning_umap_pos = hypertuning_umap_clf.fit_transform(features)
    hypertuning_umap_percentile_df = percentile_df.copy()


    hypertuning_kmeans_umap = KMeans(n_clusters=k,random_state=random_state, n_init=10)
    silhoue_scores = cross_val_score(
        hypertuning_kmeans_umap,
        hypertuning_umap_pos,
        scoring=silhouette_scorer,
        cv=3
        )
    daviesbl_scores = cross_val_score(
        hypertuning_kmeans_umap,
        hypertuning_umap_pos,
        scoring=daviesbloudin_scorer,
        cv=3
        )
    calinski_scores = cross_val_score(
        hypertuning_kmeans_umap,
        hypertuning_umap_pos,
        scoring=calinskiharabasz_scorer,
        cv=3)

    sil_scores.append(silhoue_scores.mean())
    sil_std.append(silhoue_scores.std())
    dav_scores.append(daviesbl_scores.mean())
    dav_std.append(daviesbl_scores.std())
    cal_scores.append(calinski_scores.mean())
    cal_std.append(calinski_scores.std())

hypertuning_dict = {'k': hypertuning_best_k,
                    'Silhouette Score': sil_scores,
                    'Silhouette Score Std Dev': sil_std,
                    'Davies-Bouldin Score': dav_scores,
                    'Davies-Bouldin Std Dev': dav_std,
                    'Calinski-Harabasz Score': cal_scores,
                    'Calinski-Harabasz Std Dev': cal_std,
                    }
hypertuning_dict = pd.DataFrame( hypertuning_dict, columns= hypertuning_dict.keys())
hypertuning_dict

c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs val

,k,Silhouette Score,Silhouette Score Std Dev,Davies-Bouldin Score,Davies-Bouldin Std Dev,Calinski-Harabasz Score,Calinski-Harabasz Std Dev
0,4,0.635338,0.026968,0.563242,0.049549,720.596749,85.458183
1,5,0.680127,0.027635,0.445860,0.037420,994.097280,182.498361
2,6,0.666744,0.017442,0.486902,0.017202,1227.688415,213.892535
3,7,0.632621,0.014854,0.505993,0.076268,1115.821040,146.016382
4,8,0.589166,0.016275,0.631698,0.060466,1141.592355,206.095830


In [48]:
#Hypertuning Analysis with 1 less cluster
hypertuning_best_k = (best_params['kmeans__n_clusters']-1)
hypertuning_umap_clf = umap.UMAP(
    metric=best_params['umap__metric'],
    min_dist=best_params['umap__min_dist'],
    n_neighbors=best_params['umap__n_neighbors'],
    random_state=random_state)
hypertuning_umap_pos = hypertuning_umap_clf.fit_transform(features)
hypertuning_umap_percentile_df = percentile_df.copy()

hypertuning_kmeans_umap = KMeans(
    n_clusters = hypertuning_best_k,
    random_state=random_state,
    n_init=10)
hypertuning_kmeans_umap.fit(hypertuning_umap_pos )

hypertuning_umap_percentile_df['x'] = [x[0] for x in hypertuning_umap_pos]
hypertuning_umap_percentile_df['y'] = [y[1] for y in hypertuning_umap_pos]
hypertuning_umap_percentile_df['labels'] = hypertuning_kmeans_umap.labels_

hypertuning_best_sil_score = silhouette_score(
    hypertuning_umap_pos,
    hypertuning_kmeans_umap.labels_
    )
hypertuning_best_dav_score = davies_bouldin_score(
    hypertuning_umap_pos,
    hypertuning_kmeans_umap.labels_
    )
hypertuning_best_cait_score = calinski_harabasz_score(
    hypertuning_umap_pos,
    hypertuning_kmeans_umap.labels_
    )

print(f'Silhouette score for {hypertuning_best_k} clusters: {hypertuning_best_sil_score}')
print(f'Davies-Bouldin score for {hypertuning_best_k} clusters: {hypertuning_best_dav_score}')
print(f'Calinski-Harabasz score for {hypertuning_best_k} clusters: {hypertuning_best_cait_score}')

c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1887: UserWarning: gradient function is not yet implemented for hamming distance metric; inverse_transform will be unavailable
  warn(
c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Silhouette score for 5 clusters: 0.6779688596725464
Davies-Bouldin score for 5 clusters: 0.4491275376569363
Calinski-Harabasz score for 5 clusters: 2867.1453125041508


In [49]:
#Visualization of UMAP with one less cluster
hypertuning_price_df, hypertuning_umap_price_image, hypertuning_umap_clusters, hypertuning_umap_price = genPlot(hypertuning_umap_percentile_df, False , 50)
hypertuning_umap_price_image

C:\Users\jaymj\AppData\Local\Temp\ipykernel_8116\4182680683.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.groupby('percentile').apply(lambda x: x.sample(n=sample_n, random_state=random_state)).rename(columns={'percentile':'sample_percentile'}).reset_index()


alt.LayerChart(...)

In [50]:
#Visualization of UMAP with one less cluster without images
hypertuning_umap_clusters.show()

alt.Chart(...)

In [51]:
#Focus on one specific cluster (Cluster 1 from best model) to analyze artwork
sampled_df = umap_price_df[
    (umap_price_df['x']>12)&
    (umap_price_df['x']<16)&
    (umap_price_df['y']>15)&
    (umap_price_df['y']<17)
    ]

squares = alt.Chart(sampled_df).mark_square(size=10000).encode(
x=alt.X(
    'x',
    scale=alt.Scale(
        domain=[12,16]),
    axis=alt.Axis(
        title=None,
        ticks=False,
        labelFontSize=0,
        grid=False
            )
    ),
y=alt.Y(
    'y', 
    scale=alt.Scale(domain=[15,17]), 
    axis=alt.Axis(
        title=None,
        ticks=False,
        labelFontSize=0,
        grid=False
        )
    ),
color=alt.Color(
    'percentile:N',
    legend=alt.Legend(
        title="Percentile",
        orient='top-left',
        labelFontSize=16,
        titleFontSize=20
        )
    )
)

image_url = 'image_uri_art_crop'

images = alt.Chart(sampled_df).mark_image(width=100, height=100).encode(
    x=alt.X('x'),
    y=alt.Y('y'),
    url=image_url,
    tooltip=[
        alt.Tooltip("name"),
        alt.Tooltip("subtype"),
        alt.Tooltip("keywords"),
        alt.Tooltip("color_identity"),
        alt.Tooltip("price_usd")
    ]
).properties(
    width=2000,
    height=1000
)

(squares + images)

alt.LayerChart(...)